# THME quickstart

This notebook drives THME only through its command line. Each cell runs a `thme` command with `--json` and reads the output with `json.loads`, so nothing here imports THME's Rust code.

You need the `thme` program on your PATH (see [docs/getting-started.md](../../docs/getting-started.md)). Nothing in this notebook downloads or runs a model. The notebook is saved without outputs on purpose.

In [ ]:
import json
import subprocess


## 1. Check this machine

`thme doctor --json` reports the operating system, memory, disk, GPU adapter, the models it found and the next steps. Each fact carries a `basis`: `checked` or `assumed`.

In [ ]:
doctor = subprocess.run(["thme", "doctor", "--json"], capture_output=True, text=True, check=True)
facts = json.loads(doctor.stdout)
print("keys:", sorted(facts))
print("memory total (bytes):", facts["memory_total_bytes"]["value"])
print("next steps:", facts["next"])


## 2. Find models already on this machine

`thme discover --json` lists the models from the Hugging Face cache, Ollama, LM Studio, llama.cpp and any folder you name. To search a folder of your own, add `"--path", "models"` to the command list in the next cell.

In [ ]:
found = subprocess.run(["thme", "discover", "--json"], capture_output=True, text=True, check=True)
discovered = json.loads(found.stdout)
for model in discovered["models"]:
    print(model["source"], model["format"], model["name"], model["size_bytes"], model["path"])
for warning in discovered["warnings"]:
    print("warning:", warning)


## 3. Inspect one model

Set `MODEL_PATH` to a model folder, a `.gguf` file or a `.safetensors` file. A path from the discover step works. `thme inspect` runs the safety scan and reads the model header; it does not copy the weights.

In [ ]:
from pathlib import Path

MODEL_PATH = "models/your-model"  # fill in: a folder, .gguf or .safetensors file
if not Path(MODEL_PATH).exists():
    raise FileNotFoundError(f"Set MODEL_PATH to a model on this machine first, not {MODEL_PATH!r}")

info = subprocess.run(["thme", "inspect", MODEL_PATH, "--json"], capture_output=True, text=True, check=True)
model = json.loads(info.stdout)
print(model["name"], model["architecture"], "layers:", model["num_layers"], "hidden size:", model["hidden_size"])
print("tensors:", len(model["tensors"]), "parameters:", model["num_parameters"])


## Next

`docs/notebooks.md` lists every command that prints JSON, the top-level keys of each one, and the commands that do not have `--json` yet.